# Main workflow: PE transfer learning experiment

Production notebook for full extraction + evaluation + analysis.

**Assumption:** sanity check (`01_sanity_check.ipynb`) pass.

**Workflow (in order):**
1. Setup (GPU, Drive mount, repo clone, imports)
2. Full feature extraction (12 models × 5 datasets = 60 .npz files)
3. Linear probe evaluation (60 combinations with C-grid search)
4. kNN evaluation (60 combinations with k ∈ {1, 5, 10, 20, 50})
5. Analysis of results (tables, rankings, Spearman correlations)
6. CKA evaluation + heatmap (Mechanism section + Appendix A.3)

---

## 🔧 Configuration required (Cells 1.4 and 6.1)

Before running, you must set:

- **Cell 1.4** — `CHECKPOINT_ROOT` and `DATA_HOME` (your Drive paths)
- **Cell 6.1** — `IMAGENET_TAR` (only if you run the CKA section)

All other paths in the notebook are derived from these or use ephemeral `/content/` storage; do **not** change them.

## 1. Setup

In [ ]:
# 1.1 GPU check  (no changes needed)
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 1.2 Mount Drive  (no changes needed)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1.3 Clone repo from GitHub  (no changes needed)
# Refreshed each session: code is ephemeral, lives on the Colab instance only.
import shutil, os
REPO_DST = '/content/vit-pe-transfer'
if os.path.exists(REPO_DST):
    shutil.rmtree(REPO_DST)
!git clone https://github.com/djokobandjur/vit-pe-transfer.git $REPO_DST
%cd $REPO_DST

In [ ]:
# 1.4 Path configuration
# ===========================================================================
# >>> USER CONFIGURATION REQUIRED <<<
#
# Set the TWO paths below to match YOUR Google Drive layout:
#
#   CHECKPOINT_ROOT — folder holding the 12 pretrained ViT-Base checkpoints.
#                     Expected layout:
#                         <CHECKPOINT_ROOT>/learned_seed42/best_model.pth
#                         <CHECKPOINT_ROOT>/learned_seed123/best_model.pth
#                         ... (12 total)
#                     Download the checkpoints from the link in the repo README.
#
#   DATA_HOME       — folder where this notebook will write extracted features,
#                     results JSONs, and CKA outputs. Should live on YOUR Drive
#                     so artifacts persist between sessions. Will be created if
#                     it does not exist.
# ===========================================================================
CHECKPOINT_ROOT = '/content/drive/MyDrive/Trained models_ImageNet100'    # <-- EDIT THIS
DATA_HOME       = '/content/drive/MyDrive/pe_transfer_experiment'        # <-- EDIT THIS

# ---------------------------------------------------------------------------
# Derived paths and ephemeral cache  (no changes needed below this line)
# ---------------------------------------------------------------------------
DATA_ROOT        = '/content/datasets'        # torchvision auto-download cache (ephemeral)
FEATURES_DIR     = f'{DATA_HOME}/features'
RESULTS_DIR      = f'{DATA_HOME}/results'
CKA_FEATURES_DIR = f'{DATA_HOME}/cka_features'
CKA_ANALYSIS_DIR = f'{DATA_HOME}/cka_analysis'

for d in (FEATURES_DIR, RESULTS_DIR, CKA_FEATURES_DIR, CKA_ANALYSIS_DIR):
    os.makedirs(d, exist_ok=True)

print(f'REPO_DST:         {REPO_DST}')
print(f'CHECKPOINT_ROOT:  {CHECKPOINT_ROOT}')
print(f'DATA_HOME:        {DATA_HOME}')
print(f'FEATURES_DIR:     {FEATURES_DIR}')
print(f'RESULTS_DIR:      {RESULTS_DIR}')
print(f'CKA_FEATURES_DIR: {CKA_FEATURES_DIR}')
print(f'CKA_ANALYSIS_DIR: {CKA_ANALYSIS_DIR}')

## 2. Full feature extraction

12 models × 5 datasets = 60 .npz files. The `--skip_existing` flag skips already processed combinations, allowing for a restart if the Colab session terminates.

In [ ]:
# (no changes needed)
!python -m scripts.extract_features \
    --checkpoint_root "$CHECKPOINT_ROOT" \
    --data_root "$DATA_ROOT" \
    --output_dir "$FEATURES_DIR" \
    --batch_size 128 \
    --num_workers 4 \
    --skip_existing

In [ ]:
# Verification: How many .npz files were generated?  (no changes needed)
import glob
npz_files = sorted(glob.glob(f'{FEATURES_DIR}/*.npz'))
print(f'Generated {len(npz_files)} .npz files (expected: 60)')
for f in npz_files[:5]:
    print(f'  {os.path.basename(f)}')
if len(npz_files) > 5:
    print(f'  ... ({len(npz_files) - 5} more)')

## 3. Linear probe evaluation

Logistic regression sa C-grid search (C ∈ {0.1, 1.0, 10.0}). LBFGS solver.

In [ ]:
# (no changes needed)
!python -m scripts.linear_probe \
    --features_dir "$FEATURES_DIR" \
    --output_path "$RESULTS_DIR/linear_probe.json"

## 4. kNN evaluation

kNN classification sa k ∈ {1, 5, 10, 20, 50}, cosine metric (L2-normalized features).

In [ ]:
# (no changes needed)
!python -m scripts.knn_eval \
    --features_dir "$FEATURES_DIR" \
    --output_path "$RESULTS_DIR/knn.json"

## 5. Analyze results

Aggregate u tabele: mean ± std preko seedova, per-PE rankings, cross-dataset Spearman consistency, LP vs kNN agreement.

In [ ]:
# (no changes needed)
!python -m scripts.analyze_results \
    --linear_probe "$RESULTS_DIR/linear_probe.json" \
    --knn "$RESULTS_DIR/knn.json" \
    --output_dir "$RESULTS_DIR/analysis"

## 6. CKA evaluation
- ImageNet-100 validation set setup
- CKA feature extraction (per-layer CLS activations)
- CKA computation (within-PE + cross-PE)
- Heatmap rendering (Appendix A.3 figure)

In [ ]:
# 6.1 ImageNet-100 validation set setup
# ===========================================================================
# >>> USER CONFIGURATION REQUIRED (only if you run the CKA section) <<<
# Set IMAGENET_TAR to the location of the official ILSVRC2012 validation
# tar archive on YOUR Drive. The archive must be downloaded separately from
# https://image-net.org/  (free academic registration required).
# ===========================================================================
IMAGENET_TAR = '/content/drive/MyDrive/pe_experiment/imagenet/ILSVRC2012_img_val.tar'  # <-- EDIT THIS

# ---------------------------------------------------------------------------
# Output directory and class metadata  (no changes needed)
# The class list and val→class mapping are shipped in the repo
# (data/imagenet100_classes.txt, val_labels.txt). Output goes to ephemeral
# /content/ storage because it is regenerated easily from the tar.
# ---------------------------------------------------------------------------
!python -m scripts.setup_imagenet100_val \
    --tar_path "$IMAGENET_TAR" \
    --classes_path data/imagenet100_classes.txt \
    --labels_path data/val_labels.txt \
    --output_dir /content/imagenet100/val

In [ ]:
# 6.2 CKA feature extraction — seed 0 (primary)  (no changes needed)
!python -m scripts.extract_cka_features \
    --checkpoint_root "$CHECKPOINT_ROOT" \
    --imagenet_val_root /content/imagenet100/val \
    --output_dir "$CKA_FEATURES_DIR" \
    --n_stimuli 2000 \
    --stimulus_seed 0 \
    --skip_existing

In [ ]:
# 6.3 CKA computation — seed 0  (no changes needed)
!python -m scripts.compute_cka \
    --cka_features_dir "$CKA_FEATURES_DIR" \
    --output_dir "$CKA_ANALYSIS_DIR"

In [ ]:
# 6.4 CKA replication — seed 1 (Appendix A.1)  (no changes needed)
CKA_FEATURES_SEED1 = f'{DATA_HOME}/cka_features_seed1'
CKA_ANALYSIS_SEED1 = f'{DATA_HOME}/cka_analysis_seed1'
os.makedirs(CKA_FEATURES_SEED1, exist_ok=True)
os.makedirs(CKA_ANALYSIS_SEED1, exist_ok=True)

!python -m scripts.extract_cka_features \
    --checkpoint_root "$CHECKPOINT_ROOT" \
    --imagenet_val_root /content/imagenet100/val \
    --output_dir "$CKA_FEATURES_SEED1" \
    --n_stimuli 2000 \
    --stimulus_seed 1 \
    --skip_existing

!python -m scripts.compute_cka \
    --cka_features_dir "$CKA_FEATURES_SEED1" \
    --output_dir "$CKA_ANALYSIS_SEED1"

In [ ]:
# 6.5 Heatmap rendering (Appendix A.3) — both seeds  (no changes needed)
!python -m scripts.plot_cka_heatmap \
    --cka_analysis_dir "$CKA_ANALYSIS_DIR"

!python -m scripts.plot_cka_heatmap \
    --cka_analysis_dir "$CKA_ANALYSIS_SEED1"